# SDXL Image Generator for YouTube Niche Scenes
This notebook loads SDXL on a Kaggle T4 GPU, reads `scenes.json`, generates 1920x1080 images for each scene, and packages the output as a zip file.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors torch

In [ ]:
import json
import os
import shutil
from pathlib import Path
import torch
from diffusers import StableDiffusionXLPipeline

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
scenes_path = Path('scenes.json')
output_dir = Path('sdxl_output')
output_dir.mkdir(exist_ok=True)
assert scenes_path.exists(), 'scenes.json must exist in the working directory'
with scenes_path.open('r', encoding='utf-8') as handle:
    scenes = json.load(handle)
print(f'Scenes loaded: {len(scenes)}')

In [ ]:
model_id = 'stabilityai/stable-diffusion-xl-base-1.0'
pipe = StableDiffusionXLPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to(device)
pipe.enable_attention_slicing()
pipe.enable_xformers_memory_efficient_attention()

In [ ]:
for scene in scenes:
    scene_id = scene.get('id') or scene.get('slug') or str(scene.get('index', 'unknown'))
    prompt = scene.get('prompt') or scene.get('description') or ''
    if not prompt:
        continue
    filename = output_dir / f'{scene_id}.png'
    if filename.exists():
        print('Skip existing', filename.name)
        continue
    result = pipe(prompt, height=1080, width=1920, num_inference_steps=28, guidance_scale=7.5)
    image = result.images[0]
    image.save(filename)
    print('Generated', filename.name)

In [ ]:
archive = shutil.make_archive('sdxl_images', 'zip', output_dir)
print('Packaged output:', archive)